# Un DataMart de Kimball con software libre: la medición

Cuaderno de lectura de la medición `datamart-wikipedia/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos): un laboratorio con las cuatro piezas de una plataforma analítica (RustFS 1.0.4 como almacén S3, PostgreSQL 17.11, Apache Airflow 3.3.2 y dbt-core 1.12.6 con dbt-postgres 1.11.0) sobre un dato real y público, los mil artículos más vistos de cada día en la Wikipedia en español por tipo de acceso, desde enero de 2024, tal como los sirve la API de analítica de Wikimedia. Los CSV los escribe `laboratorio.sh medir` (el medidor `medidor/medir.py` desde la red interna y `medidor/registros.py` desde el anfitrión). Este cuaderno solo los lee.

In [ ]:
import json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/datamart-wikipedia/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")

## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## Las pruebas, por bloque

In [ ]:
pr = leer("pruebas.csv")
print(len(pr), "anotaciones en", pr.bloque.nunique(), "bloques")
print(pr.groupby("bloque", sort=True).ms.agg(["count", "sum"]).rename(columns={"count": "pruebas", "sum": "ms"}))

## Tamaños: cada zona y cada tabla

In [ ]:
t = leer("tamanos.csv")
t["MB"] = (t.octetos / 1e6).round(1)
print(t[["zona", "objeto", "filas", "MB"]].to_string(index=False))
print("total PostgreSQL:", round(t[t.zona == "postgresql"].octetos.sum() / 1e6, 1), "MB ·", "total S3:", round(t[t.zona == "s3"].octetos.sum() / 1e6, 1), "MB")

## Cargas: un año de filas, cinco maneras

In [ ]:
cg = leer("cargas.csv")
cg["s"] = (cg.ms / 1000).round(1)
print(cg[["metodo", "filas", "s", "filas_por_s"]].to_string(index=False))
ax = cg.plot.barh(x="metodo", y="filas_por_s", legend=False, figsize=(8, 3.2), color="#2a78d6")
ax.invert_yaxis(); ax.set_xlabel("filas por segundo"); ax.set_ylabel(""); plt.tight_layout(); plt.show()

## Consultas: plano, estrella y copo de nieve, con y sin índices

In [ ]:
cs = leer("consultas.csv")
tabla = cs.pivot_table(index="consulta", columns=["indices", "modelo"], values="ms").round(1)
print(tabla.to_string())
print()
print("suma de las diez (ms):")
print(cs.groupby(["indices", "modelo"]).ms.sum().round(0).unstack())
print("octetos por modelo:", cs.groupby("modelo").octetos_modelo.first().to_dict())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, idx in zip(axes, (0, 1)):
    sub = cs[cs.indices == idx].pivot(index="consulta", columns="modelo", values="ms")[["plano", "estrella", "copo"]]
    sub.plot.barh(ax=ax, color=["#898781", "#2a78d6", "#e08a3c"])
    ax.invert_yaxis(); ax.set_title("con índices" if idx else "sin índices"); ax.set_xlabel("ms (mediana de 5)"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

## Estrategias incrementales de dbt

In [ ]:
es = leer("estrategias.csv")
print(es[["estrategia", "codigo", "segundos", "filas_despues", "claves_duplicadas"]].to_string(index=False))

## Tests de dbt

In [ ]:
ts = leer("tests_dbt.csv")
print(len(ts), "tests:", ts.estado.value_counts().sort_index().sort_values(ascending=False, kind="stable").to_dict())
print(ts[ts.estado != "pass"][["test", "estado", "filas_fallidas"]].to_string(index=False))

## Airflow: el relleno histórico y los fallos simulados

In [ ]:
rn = leer("runs_airflow.csv"); ta = leer("tareas_airflow.csv")
prog = rn[rn.tipo == "scheduled"]; man = rn[rn.tipo != "scheduled"]
print(len(prog), "runs programados:", prog.estado.value_counts().sort_index().sort_values(ascending=False, kind="stable").to_dict(),
      "· mediana", round(prog.segundos.median(), 1), "s · suma", round(prog.segundos.sum()), "s")
print(len(man), "runs manuales (fallo simulado):", man[["run_id", "estado", "segundos"]].to_string(index=False, header=False))
ok = ta[(ta.estado == "success") & ta.run_id.str.startswith("scheduled")]
print("mediana de segundos por tarea (runs programados):")
print(ok.groupby("tarea").segundos.median().round(1).sort_values(ascending=False))
print(ta[~ta.run_id.str.startswith("scheduled")][["run_id", "tarea", "estado", "intentos", "segundos"]].to_string(index=False))
print(leer("avisos_airflow.csv").to_string(index=False))

In [ ]:
ok2 = ta[ta.run_id.str.startswith("scheduled")].pivot_table(index="mes", columns="tarea", values="segundos", aggfunc="sum")
cols = [c for c in ["descargar", "subir", "preparar", "cargar", "dbt_seed", "dbt_run", "dbt_test", "dbt_snapshot"] if c in ok2.columns]
ax = ok2[cols].plot.bar(stacked=True, figsize=(11, 4), width=0.85)
ax.set_ylabel("segundos"); ax.set_xlabel(""); ax.legend(ncol=4, fontsize=8); plt.tight_layout(); plt.show()

## Planes de la consulta de los diez más vistos

In [ ]:
pl = leer("planes.csv")
print(pl[["modelo", "indices", "ms_ejecucion", "ms_planificacion", "nodo_raiz", "nodo_peor_estimado", "relacion", "filas_estimadas", "filas_reales", "bloques_compartidos"]].to_string(index=False))

## La dimensión con historia: hora de observación frente a fecha de negocio

In [ ]:
sn = leer("snapshot.csv"); print(sn.to_string(index=False))
sv = leer("snapshot_versiones.csv"); print(sv.to_string(index=False))
se = leer("snapshot_ejemplo.csv"); print(len(se), "versiones del artículo con más cambios:"); print(se[["tramo", "fecha_datos", "dbt_valid_from", "dbt_valid_to"]].to_string(index=False))

## El ranking derivado frente a la lista mensual oficial

In [ ]:
rk = leer("ranking_oficial.csv")
print(rk.to_string(index=False))
print("medianas:", rk[["coincidencias_10", "coincidencias_100", "coincidencias_1000", "ausentes_en_derivado", "mediana_cociente_visitas_top100"]].median().round(3).to_dict())
ax = rk.plot(x="anio_mes", y=["coincidencias_100", "coincidencias_1000"], figsize=(10, 3.4), color=["#2a78d6", "#e08a3c"])
ax.set_ylabel("artículos coincidentes"); ax.set_xlabel(""); plt.tight_layout(); plt.show()

## Idempotencia: octubre cargado dos veces

In [ ]:
print(leer("idempotencia.csv").to_string(index=False))

## Qué cuesta

In [ ]:
e = leer("esfuerzo.csv"); cf = leer("configuracion.csv"); im = leer("imagenes.csv"); rc = leer("recursos.csv")
print("init:", int(e.segundos_acumulados.iloc[-1]), "s ·", "; ".join(f"{f} a los {s} s" for f, s in zip(e.fase, e.segundos_acumulados)))
print(int(cf.lineas.sum()), "líneas de configuración, modelos y guiones en", len(cf), "ficheros")
print(cf.sort_values("lineas", ascending=False).head(8).to_string(index=False))
print(im.to_string(index=False)); print(rc.to_string(index=False))